# Exercise 4: Object Detection — From R-CNN to YOLO
## How Computers Learn to Find Objects in Images

---

### Learning Objectives
- Understand the evolution from R-CNN → Fast R-CNN → Faster R-CNN → YOLO
- Implement **Intersection over Union (IoU)** and **Non-Maximum Suppression (NMS)** from scratch
- Run Faster R-CNN and YOLO on real images and compare speed vs accuracy
- Understand what makes YOLO "real-time" while R-CNN family is slow

### Architecture Family Tree
```
R-CNN (2013) — selective search + CNN per region — ~47 sec/image
  └→ Fast R-CNN (2015) — ROI pooling, 1 CNN pass — ~2 sec/image
       └→ Faster R-CNN (2015) — RPN replaces selective search — ~200ms/image

YOLO (2016) — single-pass grid, real-time — ~25ms/image
```

---

## Google Colab Setup

> **Before running:** Go to `Runtime → Change runtime type → T4 GPU` to enable GPU acceleration.
>
> Then run the **Setup** cell below, followed by **Run All** (`Runtime → Run all`).

In [ ]:
# Install YOLOv8 (not pre-installed in Colab)
!pip install -q ultralytics

# Verify GPU and core packages
import torch, torchvision
print(f'PyTorch {torch.__version__}  |  torchvision {torchvision.__version__}')
print(f'GPU available: {torch.cuda.is_available()}  — device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
from ultralytics import YOLO
print('ultralytics / YOLOv8 ready!')

In [ ]:
import torch
import torchvision
import torchvision.transforms as T
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import requests
from io import BytesIO
import time

plt.style.use('seaborn-v0_8-white')

COCO_LABELS = [
    '__background__', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A', 'stop sign',
    'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow',
    'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack', 'umbrella', 'N/A', 'N/A',
    'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball',
    'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket',
    'bottle', 'N/A', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl',
    'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza',
    'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table',
    'N/A', 'N/A', 'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard',
    'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'N/A', 'book',
    'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'
]

COLORS = plt.cm.Set3(np.linspace(0, 1, 91))
print('Setup complete!')

## Part 1: The Building Blocks — Implement IoU and NMS from Scratch

**IoU (Intersection over Union)** measures how well two boxes overlap:
$$IoU = \frac{\text{Area of Intersection}}{\text{Area of Union}}$$

**NMS (Non-Maximum Suppression)** removes duplicate detections.

In [ ]:
def compute_iou(box1, box2):
    """
    Compute IoU between two boxes.
    boxes in format [x1, y1, x2, y2]
    """
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection
    
    return intersection / (union + 1e-6)


def non_maximum_suppression(boxes, scores, iou_threshold=0.5):
    """
    Remove duplicate detections.
    Returns indices of boxes to keep.
    """
    if len(boxes) == 0:
        return []
    
    # Sort by confidence score (highest first)
    order = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    keep = []
    
    while order:
        best = order.pop(0)
        keep.append(best)
        order = [i for i in order if compute_iou(boxes[best], boxes[i]) < iou_threshold]
    
    return keep


# === Visualize IoU ===
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

scenarios = [
    ([1, 1, 4, 4], [2, 2, 5, 5], 'Partial Overlap'),
    ([1, 1, 4, 4], [1, 1, 4, 4], 'Perfect Match (IoU=1)'),
    ([1, 1, 2, 2], [3, 3, 5, 5], 'No Overlap (IoU=0)'),
]

for ax, (b1, b2, title) in zip(axes, scenarios):
    iou = compute_iou(b1, b2)
    for box, color, label in [(b1, '#3498db', 'Predicted'), (b2, '#e74c3c', 'Ground Truth')]:
        rect = patches.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1],
                                  linewidth=3, edgecolor=color, facecolor=color, alpha=0.3, label=label)
        ax.add_patch(rect)
    ax.set_xlim(0, 7); ax.set_ylim(0, 7)
    ax.set_aspect('equal')
    ax.set_title(f'{title}\nIoU = {iou:.3f}', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)

plt.suptitle('Intersection over Union (IoU)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


# === Visualize NMS ===
np.random.seed(42)
# Simulate a car with 8 overlapping detections and some false positives
boxes_raw = [
    [100, 150, 300, 280],  # True car
    [105, 155, 305, 285],  # Duplicate
    [98,  148, 298, 278],  # Duplicate
    [110, 160, 310, 290],  # Duplicate
    [102, 152, 302, 282],  # Duplicate
    [400, 100, 550, 200],  # Second car
    [398,  98, 548, 198],  # Duplicate
    [50,  50, 150, 150],   # Background (low score)
]
scores_raw = [0.95, 0.87, 0.82, 0.79, 0.76, 0.90, 0.85, 0.42]

kept = non_maximum_suppression(boxes_raw, scores_raw, iou_threshold=0.5)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, (boxes, title) in zip(axes, [
    (range(len(boxes_raw)), f'Before NMS ({len(boxes_raw)} detections)'),
    (kept, f'After NMS ({len(kept)} detections)'),
]):
    ax.set_facecolor('#ecf0f1')
    ax.set_xlim(0, 600); ax.set_ylim(0, 350)
    for i in boxes:
        b = boxes_raw[i]
        color = '#2ecc71' if i in kept else '#e74c3c'
        rect = patches.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1],
                                  linewidth=2.5, edgecolor=color, facecolor=color, alpha=0.2)
        ax.add_patch(rect)
        ax.text(b[0]+2, b[1]+12, f'{scores_raw[i]:.2f}', color=color, fontweight='bold', fontsize=9)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.invert_yaxis()

plt.suptitle('Non-Maximum Suppression (NMS)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 2: Faster R-CNN in Action

Faster R-CNN uses a **Region Proposal Network (RPN)** — a small CNN that slides over the feature map and proposes where objects might be. These proposals are then classified by a second head.

Key insight: The RPN shares convolutional features with the detection head → fast!

In [ ]:
# Load pretrained Faster R-CNN
weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
faster_rcnn = fasterrcnn_resnet50_fpn(weights=weights)
faster_rcnn.eval()
print('Faster R-CNN loaded!')
print(f'Parameters: {sum(p.numel() for p in faster_rcnn.parameters()):,}')

In [ ]:
def load_sample_image(url=None, path=None):
    """Load an image from URL or path."""
    if url:
        response = requests.get(url, timeout=10)
        img = Image.open(BytesIO(response.content)).convert('RGB')
    else:
        img = Image.open(path).convert('RGB')
    return img

# Use a local test image or download one
# Option 1: Use torchvision's sample image
import torchvision.io as io

# Create a simple test image using torchvision's dog sample
from torchvision.utils import draw_bounding_boxes

# Download and use a sample COCO-style image
img_url = 'http://images.cocodataset.org/val2017/000000039769.jpg'
try:
    img = load_sample_image(url=img_url)
    print('Downloaded COCO sample image (cats on couch)')
except:
    # Fallback: generate a synthetic image
    img = Image.fromarray(np.random.randint(100, 200, (480, 640, 3), dtype=np.uint8))
    print('Using fallback synthetic image')

transform = T.Compose([T.ToTensor()])
img_tensor = transform(img)

# Run Faster R-CNN
with torch.no_grad():
    t0 = time.time()
    predictions = faster_rcnn([img_tensor])
    t_frcnn = time.time() - t0

pred = predictions[0]
confidence_threshold = 0.7
mask = pred['scores'] > confidence_threshold

boxes   = pred['boxes'][mask].numpy()
labels  = pred['labels'][mask].numpy()
scores  = pred['scores'][mask].numpy()

print(f'Faster R-CNN inference: {t_frcnn*1000:.1f} ms')
print(f'Objects detected (conf > {confidence_threshold}): {len(boxes)}')
for box, label, score in zip(boxes, labels, scores):
    print(f'  {COCO_LABELS[label]:15s} | score: {score:.3f} | box: [{box[0]:.0f}, {box[1]:.0f}, {box[2]:.0f}, {box[3]:.0f}]')

# Visualize
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
ax.imshow(img)
for box, label, score in zip(boxes, labels, scores):
    color = COLORS[label]
    rect = patches.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1],
                               linewidth=2.5, edgecolor=color, facecolor='none')
    ax.add_patch(rect)
    ax.text(box[0], box[1]-5, f'{COCO_LABELS[label]} {score:.2f}',
            color='white', fontsize=11, fontweight='bold',
            bbox=dict(facecolor=color, alpha=0.8, pad=2, edgecolor='none'))

ax.set_title(f'Faster R-CNN Detections — {t_frcnn*1000:.0f}ms | {len(boxes)} objects found', 
             fontsize=13, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

## Part 3: YOLO — You Only Look Once

YOLO's key insight: frame detection as a **regression problem**. Divide the image into an S×S grid. Each cell predicts B bounding boxes and C class probabilities simultaneously in a single forward pass.

No region proposals. No two-stage pipeline. **One shot = one answer.**

In [ ]:
# YOLO grid visualization — understanding the core idea
def visualize_yolo_grid(img, S=7):
    """Show how YOLO divides the image into a grid."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    # Left: show the grid overlay
    axes[0].imshow(img)
    h, w = np.array(img).shape[:2]
    
    # Draw grid
    for i in range(1, S):
        axes[0].axhline(i * h / S, color='yellow', linewidth=1, alpha=0.7)
        axes[0].axvline(i * w / S, color='yellow', linewidth=1, alpha=0.7)
    
    # Highlight a responsible cell (e.g., center of the cat)
    cx, cy = w * 0.3, h * 0.4
    cell_x = int(cx / (w / S)) * (w / S)
    cell_y = int(cy / (h / S)) * (h / S)
    rect = patches.Rectangle((cell_x, cell_y), w/S, h/S, linewidth=3, edgecolor='#e74c3c', facecolor='#e74c3c', alpha=0.3)
    axes[0].add_patch(rect)
    axes[0].set_title(f'YOLO {S}×{S} Grid\n(red cell is responsible for detecting the object whose center falls in it)', 
                      fontsize=11, fontweight='bold')
    axes[0].axis('off')
    
    # Right: show what each grid cell predicts
    axes[1].set_facecolor('#2c3e50')
    axes[1].set_xlim(0, 10); axes[1].set_ylim(0, 12)
    
    items = [
        (8, 11, '#e74c3c', 'tx, ty — center offset within cell'),
        (8, 9.5, '#e74c3c', 'tw, th — box width/height'),
        (8, 8, '#f39c12', 'objectness score P(object)'),
        (8, 6, '#2ecc71', 'class probabilities\nP(class | object)'),
    ]
    
    for x, y, color, label in items:
        axes[1].plot(0.5, y, 's', color=color, markersize=12)
        axes[1].text(1.2, y, label, color='white', fontsize=11, va='center')
    
    axes[1].set_title('Each Grid Cell Predicts:', color='white', fontsize=13, fontweight='bold', pad=15)
    axes[1].text(5, 4, f'= {5 * 2 + 80} numbers per cell', color='#bdc3c7', fontsize=12, ha='center')
    axes[1].text(5, 2.5, f'Total output: {S}×{S}×(5×B + C) = {S*S*(5*2+80)}', color='#bdc3c7', fontsize=11, ha='center')
    axes[1].text(5, 1.2, '(for 2 boxes, 80 COCO classes)', color='#7f8c8d', fontsize=10, ha='center')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()

visualize_yolo_grid(img, S=7)

In [ ]:
# === Run YOLO (via ultralytics if available) ===
try:
    from ultralytics import YOLO
    yolo_model = YOLO('yolov8n.pt')
    
    img_array = np.array(img)
    t0 = time.time()
    yolo_results = yolo_model(img_array, verbose=False)
    t_yolo = time.time() - t0
    
    print(f'YOLOv8n inference: {t_yolo*1000:.1f} ms')
    print(f'Faster R-CNN:      {t_frcnn*1000:.1f} ms')
    print(f'Speedup: {t_frcnn/t_yolo:.1f}× faster')
    
    # Plot YOLO results
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    axes[0].imshow(img)
    for box, label, score in zip(boxes, labels, scores):
        color = COLORS[label]
        rect = patches.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1],
                                   linewidth=2.5, edgecolor=color, facecolor='none')
        axes[0].add_patch(rect)
        axes[0].text(box[0], box[1]-5, f'{COCO_LABELS[label]} {score:.2f}',
                    color='white', fontsize=10, fontweight='bold',
                    bbox=dict(facecolor=color, alpha=0.8, pad=2, edgecolor='none'))
    axes[0].set_title(f'Faster R-CNN\n{t_frcnn*1000:.0f}ms | {len(boxes)} detections', fontsize=12, fontweight='bold')
    axes[0].axis('off')
    
    yolo_result_img = yolo_results[0].plot()
    axes[1].imshow(yolo_result_img[:,:,::-1])
    n_yolo_det = len(yolo_results[0].boxes)
    axes[1].set_title(f'YOLOv8n\n{t_yolo*1000:.0f}ms | {n_yolo_det} detections', fontsize=12, fontweight='bold')
    axes[1].axis('off')
    
    plt.suptitle('Faster R-CNN vs YOLOv8 — Speed-Accuracy Tradeoff', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

except ImportError:
    print('ultralytics not installed. Install with: pip install ultralytics')
    print('Showing Faster R-CNN results only.')

## Part 4: Speed vs Accuracy — The Core Tradeoff

Different architectures occupy different points on the Pareto frontier.

In [ ]:
# Approximate published benchmarks (COCO test-dev, V100 GPU)
models_info = {
    'R-CNN\n(2013)':          {'map': 53.7, 'fps': 0.02,  'color': '#e74c3c',  'size': 200},
    'Fast R-CNN\n(2015)':     {'map': 66.9, 'fps': 0.5,   'color': '#e67e22',  'size': 200},
    'Faster RCNN\nResNet50':  {'map': 37.0, 'fps': 5,     'color': '#f39c12',  'size': 200},
    'YOLOv3':                 {'map': 33.0, 'fps': 45,    'color': '#27ae60',  'size': 200},
    'YOLOv5s':                {'map': 37.4, 'fps': 100,   'color': '#2ecc71',  'size': 200},
    'YOLOv8n':                {'map': 37.3, 'fps': 160,   'color': '#1abc9c',  'size': 200},
    'YOLOv8x':                {'map': 53.9, 'fps': 55,    'color': '#16a085',  'size': 200},
    'DETR':                   {'map': 42.0, 'fps': 28,    'color': '#9b59b6',  'size': 200},
}

fig, ax = plt.subplots(figsize=(12, 7))

for name, info in models_info.items():
    ax.scatter(info['fps'], info['map'], s=info['size'], c=info['color'], 
               alpha=0.85, edgecolors='white', linewidths=1.5, zorder=5)
    offset_x = 2 if info['fps'] < 80 else -5
    offset_y = 0.5
    ax.annotate(name, (info['fps'], info['map']), 
                xytext=(info['fps'] + offset_x, info['map'] + offset_y),
                fontsize=10, fontweight='bold', ha='left')

ax.set_xlabel('Inference Speed (FPS on GPU)', fontsize=13)
ax.set_ylabel('COCO mAP (%)', fontsize=13)
ax.set_title('Object Detection: Speed vs Accuracy Pareto Frontier', fontsize=14, fontweight='bold')
ax.set_xscale('log')

# Add regions
ax.axvspan(0, 1, alpha=0.05, color='red', label='Too slow for real-time')
ax.axvspan(25, 200, alpha=0.05, color='green', label='Real-time (≥25 FPS)')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

## Exercises

### Exercise A — Implement a Simple Anchor Generator
Faster R-CNN uses **anchor boxes** — pre-defined aspect ratios and scales placed at every feature map location. Implement an anchor generator:
```python
def generate_anchors(feature_map_size, scales=[32, 64, 128], ratios=[0.5, 1.0, 2.0]):
    """Generate all anchors for a given feature map.
    Returns: Tensor of shape [H*W*num_anchors, 4] in (x1,y1,x2,y2) format
    """
    # YOUR CODE HERE
```
Visualize the anchors for a 7×7 feature map on a 224×224 image.

### Exercise B — YOLO Loss Function
YOLO's loss is a sum of three terms:
$$L = \lambda_{coord} L_{box} + L_{obj} + \lambda_{noobj} L_{noobj} + L_{cls}$$

Implement each term for a single image prediction. Why is $\lambda_{coord}=5$ and $\lambda_{noobj}=0.5$?

### Exercise C — Live Webcam Detection (Bonus)
If you have a webcam, run YOLOv8 in real-time:
```python
import cv2
cap = cv2.VideoCapture(0)
model = YOLO('yolov8n.pt')
while True:
    ret, frame = cap.read()
    results = model(frame)
    cv2.imshow('YOLO Real-time', results[0].plot())
    if cv2.waitKey(1) == ord('q'):
        break
```

### Discussion Questions
1. Faster R-CNN is a two-stage detector. What happens in each stage? Why does this make it more accurate but slower?
2. YOLO struggles with small, closely-packed objects (e.g., a flock of birds). Why? What architectural change in YOLOv2+ addressed this?
3. You're building a system to detect defects on a factory production line running at 60 frames/second. Which model would you choose and why?